## 01. KB 데이터 탐색 (EDA)

KB부동산 통계 테이블의 구조, 날짜 범위, 지역코드 형식을 확인합니다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가 (notebooks/ -> part08_kb_dashboard/ -> projects/ -> 프로젝트 루트)
PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from projects.part08_kb_dashboard.dashboard.db import execute_query

### 1. KB 테이블 목록 및 날짜 범위

In [ ]:
# KB 3개 통계 테이블의 날짜 범위 확인
tables = ['KB_매매가격지수', 'KB_전세가격지수', 'KB_전세가율']
for t in tables:
    df = execute_query(f"""
        SELECT
            COUNT(*) AS 행수,
            MIN(날짜) AS 시작일,
            MAX(날짜) AS 종료일
        FROM {t}
        WHERE 매물종별구분 = '아파트'
          AND 월간주간구분 = '월간'
    """)
    row = df.iloc[0]
    print(f"{t}: {row['행수']:,}행 ({row['시작일']} ~ {row['종료일']})")

### 2. KB_매매가격지수 컬럼 구조

In [ ]:
# 컬럼 구조 확인
df = execute_query("DESCRIBE KB_매매가격지수")
display(df)

In [ ]:
# 샘플 데이터 (2025년 9월, 아파트, 월간)
df = execute_query("""
    SELECT *
    FROM KB_매매가격지수
    WHERE 날짜 = '2025-09-01'
      AND 매물종별구분 = '아파트'
      AND 월간주간구분 = '월간'
    LIMIT 10
""")
display(df)

### 3. 지역코드 형식 분석

KB 지역코드는 10자리 구조다. 앞 5자리가 시군구 코드(행정구역 코드), 뒤 5자리는 KB 내부 분류 코드다.
뒤 8자리가 `00000000`인 경우 시도 전체 집계이므로 제외한다.

In [ ]:
# 지역코드 길이별 건수
df = execute_query("""
    SELECT
        LENGTH(지역코드) AS 코드길이,
        COUNT(DISTINCT 지역코드) AS 고유코드수
    FROM KB_매매가격지수
    WHERE 매물종별구분 = '아파트'
      AND 월간주간구분 = '월간'
      AND 날짜 = '2025-09-01'
    GROUP BY 코드길이
    ORDER BY 코드길이
""")
display(df)

In [ ]:
# 시군구 단위 코드 (10자리, 뒤 8자리 ≠ 00000000) 확인
df = execute_query("""
    SELECT DISTINCT
        지역코드,
        지역명,
        LEFT(지역코드, 5) AS sig_cd
    FROM KB_매매가격지수
    WHERE 날짜 = '2025-09-01'
      AND 매물종별구분 = '아파트'
      AND 월간주간구분 = '월간'
      AND LENGTH(지역코드) = 10
      AND RIGHT(지역코드, 8) != '00000000'
    ORDER BY 지역코드
    LIMIT 20
""")
display(df)

In [ ]:
# 시군구 단위 코드 총 개수
df = execute_query("""
    SELECT COUNT(DISTINCT LEFT(지역코드, 5)) AS 시군구수
    FROM KB_매매가격지수
    WHERE 날짜 = '2025-09-01'
      AND 매물종별구분 = '아파트'
      AND 월간주간구분 = '월간'
      AND LENGTH(지역코드) = 10
      AND RIGHT(지역코드, 8) != '00000000'
""")
print(f"KB 통계 대상 시군구 수: {df.iloc[0]['시군구수']}개")

### 4. 매물종별구분 및 월간주간구분 분포

In [ ]:
# 매물종별구분 종류
df = execute_query("""
    SELECT 매물종별구분, COUNT(DISTINCT 날짜) AS 월수
    FROM KB_매매가격지수
    GROUP BY 매물종별구분
    ORDER BY 월수 DESC
""")
display(df)

In [ ]:
# 월간주간구분 종류
df = execute_query("""
    SELECT 월간주간구분, COUNT(*) AS cnt
    FROM KB_매매가격지수
    WHERE 매물종별구분 = '아파트'
    GROUP BY 월간주간구분
""")
display(df)

### 5. 매매 테이블 거래량 데이터 (Part08에서 사용)

KB 통계에 거래량이 없어서 실거래가 테이블(매매)에서 시군구별로 집계한다.

In [ ]:
# 2025년 9월 시군구별 거래량 상위 10개
df = execute_query("""
    SELECT
        CASE
            WHEN 시도 = '세종특별자치시' THEN '세종특별자치시'
            ELSE 시군구
        END AS sig_kor_nm,
        COUNT(*) AS 거래량
    FROM 매매
    WHERE 계약년월 = 202509
      AND 거래유형 != '직거래'
      AND (해제사유발생일 IS NULL OR 해제사유발생일 IN ('None', '-'))
    GROUP BY sig_kor_nm
    ORDER BY 거래량 DESC
    LIMIT 10
""")
display(df)